# 2 · 3D Descriptor Calculator (standalone)

**Chameleon Predictor reproducibility notebook — 2 of 3**  
*generate → **calculate** → plot*

Point this at the ensembles from Notebook 1 (an **R** run folder and an **S** run folder) and it writes two descriptor files:

1. **`per_conformer_<pair>.csv`** — one row per conformer; the file **Notebook 3 reads**.
2. **`descriptors_<pair>.csv`** — a **geometric summary** per isomer / solvent / descriptor (median, min, max, range, IQR) + the median ΔPSA.

**Geometry-first, full ensemble.** It reads the **full native `ensemble.xyz`** — *all* of CREST's cregen conformers (e.g. chloroform 530, not the reduced-to-50 `ensemble.sdf/json`) — and summarizes their **distribution**. There is **no local dedup**: conformer granularity is set upstream at cregen (`-ewin`/`-rmsd`). Energy/weight are retained only as metadata. **Standalone:** the descriptor math is inlined below — no `scripts/` import.

## Requirements
Any machine with **`rdkit`, `numpy`, `pandas`** and a Jupyter kernel — no external binaries, no `scripts/` folder. On this repository the ready kernel is the **`base`** conda env.

```
conda create -n chameleon -c conda-forge python=3.11 rdkit numpy pandas jupyter ipykernel
```

## Inlined descriptor library
The 3D-descriptor functions — the solvent-accessible **3D-PSA of record** (Ono 2019 / Begnini 2021: N/O + polar H, Ertl sulfur rule), the **intramolecular H-bond** breakdown (backbone vs side-chain), and the surface/shape terms — copied from `phys_descriptors_v3.py` so this notebook stands alone. Run this cell once; the workflow below uses these functions.

> If the science changes, edit the repo script and re-inline — this cell is a copy, not a live import.

In [1]:
import json
import numpy as np
import pandas as pd
from pathlib import Path
from rdkit import Chem, RDLogger
from rdkit.Chem import Descriptors3D
RDLogger.DisableLog('rdApp.*')

# Bondi van der Waals radii (Å) and atom-class sets
_BONDI = {"H": 1.20, "C": 1.70, "N": 1.55, "O": 1.52, "S": 1.80,
          "P": 1.80, "F": 1.47, "Cl": 1.75, "Br": 1.85, "I": 1.98}
_POLAR = {"N", "O", "S", "P"}
_DONOR_HEAVY = {"N", "O"}
_ACCEPTOR = {"N", "O"}
_HB_DIST_MAX = 2.5
_HB_ANGLE_MIN = 120.0
SOLVENTS = ["water", "mem"]


# ── 3D descriptors (copied from phys_descriptors_v3) ──────────────────────────
def _per_atom_sasa(mol, conf_id):
    """Per-atom SASA (Å²) via rdFreeSASA with Bondi radii; None on failure."""
    from rdkit.Chem import rdFreeSASA
    try:
        radii = [_BONDI.get(a.GetSymbol(), 1.50) for a in mol.GetAtoms()]
        rdFreeSASA.CalcSASA(mol, radii, confIdx=conf_id)
        return np.array([float(a.GetPropsAsDict().get("SASA", 0.0))
                         for a in mol.GetAtoms()], dtype=float)
    except Exception:
        return None


def _atom_classes(mol):
    n = mol.GetNumAtoms()
    is_polar = np.zeros(n, dtype=bool); is_donor_h = np.zeros(n, dtype=bool)
    is_apolar = np.zeros(n, dtype=bool); is_acceptor = np.zeros(n, dtype=bool)
    for atom in mol.GetAtoms():
        i = atom.GetIdx(); sym = atom.GetSymbol()
        if sym in _POLAR:
            is_polar[i] = True
            if sym in _ACCEPTOR:
                is_acceptor[i] = True
        elif sym == "C":
            is_apolar[i] = True
        elif sym == "H":
            nbrs = atom.GetNeighbors()
            if nbrs:
                hsym = nbrs[0].GetSymbol()
                if hsym in _DONOR_HEAVY:
                    is_donor_h[i] = True
                elif hsym == "C":
                    is_apolar[i] = True
    return is_polar, is_donor_h, is_apolar, is_acceptor


def _amphi_moment(coords, sasa, is_polar, is_apolar):
    wp = sasa * is_polar; wa = sasa * is_apolar
    sp, sa = wp.sum(), wa.sum()
    if sp <= 0 or sa <= 0:
        return float("nan")
    c_polar = (coords * wp[:, None]).sum(axis=0) / sp
    c_apolar = (coords * wa[:, None]).sum(axis=0) / sa
    return float(np.linalg.norm(c_polar - c_apolar))


def surface_descriptors_mol(mol, conf_id=-1):
    """3D-PSA of record + surface terms (N/O + polar H + oxidized S only; Ertl sulfur rule)."""
    if conf_id == -1:
        conf_id = mol.GetConformer().GetId()
    nan = float("nan")
    out = {"psa": nan, "hbd_sasa": nan, "hba_sasa": nan, "hydrophobic_sasa": nan,
           "total_sasa": nan, "amphi_moment": nan}
    sasa = _per_atom_sasa(mol, conf_id)
    if sasa is None:
        return out
    is_polar, is_donor_h, is_apolar, is_acceptor = _atom_classes(mol)
    oxidized_s = np.array([
        a.GetSymbol() == "S" and any(
            b.GetBondTypeAsDouble() == 2.0 and b.GetOtherAtom(a).GetSymbol() == "O"
            for b in a.GetBonds())
        for a in mol.GetAtoms()])
    out["total_sasa"] = round(float(sasa.sum()), 2)
    out["psa"] = round(float(sasa[is_acceptor | is_donor_h | oxidized_s].sum()), 2)
    out["hbd_sasa"] = round(float(sasa[is_donor_h].sum()), 2)
    out["hba_sasa"] = round(float(sasa[is_acceptor].sum()), 2)
    out["hydrophobic_sasa"] = round(float(sasa[is_apolar].sum()), 2)
    coords = mol.GetConformer(conf_id).GetPositions()
    out["amphi_moment"] = round(_amphi_moment(coords, sasa, is_polar, is_apolar), 3)
    return out


def macrocycle_atoms(mol):
    rings = mol.GetRingInfo().AtomRings()
    return set(max(rings, key=len)) if rings else set()


def backbone_hbond_atoms(mol):
    ring = macrocycle_atoms(mol); bb = set(ring)
    for atom in mol.GetAtoms():
        if atom.GetSymbol() == "O" and any(nb.GetIdx() in ring for nb in atom.GetNeighbors()):
            bb.add(atom.GetIdx())
    return bb


def imhb_descriptors_mol(mol, conf_id=-1, backbone_atoms=None):
    if conf_id == -1:
        conf_id = mol.GetConformer().GetId()
    if backbone_atoms is None:
        backbone_atoms = backbone_hbond_atoms(mol)
    coords = mol.GetConformer(conf_id).GetPositions()
    donors, acceptors = [], []
    for atom in mol.GetAtoms():
        sym = atom.GetSymbol()
        if sym in _ACCEPTOR:
            acceptors.append(atom.GetIdx())
        elif sym == "H":
            nbrs = atom.GetNeighbors()
            if nbrs and nbrs[0].GetSymbol() in _DONOR_HEAVY:
                donors.append((atom.GetIdx(), nbrs[0].GetIdx()))
    imhb = bb = res = 0
    donor_set, acc_set = set(), set()
    for h_idx, d_idx in donors:
        h, d = coords[h_idx], coords[d_idx]
        for a_idx in acceptors:
            if a_idx == d_idx:
                continue
            a = coords[a_idx]
            if np.linalg.norm(h - a) > _HB_DIST_MAX:
                continue
            v1, v2 = d - h, a - h
            cos = np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2) + 1e-10)
            if np.degrees(np.arccos(np.clip(cos, -1, 1))) < _HB_ANGLE_MIN:
                continue
            imhb += 1; donor_set.add(h_idx); acc_set.add(a_idx)
            if d_idx in backbone_atoms and a_idx in backbone_atoms:
                bb += 1
            else:
                res += 1
    return {"imhb": imhb, "imhbd": len(donor_set), "imhba": len(acc_set),
            "imhb_bb": bb, "imhb_res": res}


# ── Full native ensemble I/O ──────────────────────────────────────────────────
# We read ensemble.xyz — CREST's full cregen conformer set (copied before any reduction) —
# so the analysis sees ALL cregen-unique conformers (e.g. chloroform 530, not the capped 50).
# No local RMSD dedup here: conformer granularity is set at cregen (-ewin/-rmsd), by design.
def parse_xyz_ensemble(xyz_path):
    """Parse a multi-conformer XYZ; energy read from each conformer's comment line.
    Returns [(symbols, coords, energy), ...]."""
    conformers = []
    lines = Path(xyz_path).read_text().splitlines()
    i = 0
    while i < len(lines):
        line = lines[i].strip()
        if not line:
            i += 1; continue
        try:
            n_atoms = int(line)
        except ValueError:
            i += 1; continue
        i += 1
        energy = np.nan
        if i < len(lines):
            for tok in lines[i].replace('=', ' ').replace(':', ' ').split():
                try:
                    energy = float(tok); break
                except ValueError:
                    continue
        i += 1
        symbols, coords = [], []
        for _ in range(n_atoms):
            if i >= len(lines):
                break
            parts = lines[i].split()
            if len(parts) >= 4:
                symbols.append(parts[0])
                coords.append([float(parts[1]), float(parts[2]), float(parts[3])])
            i += 1
        if len(symbols) == n_atoms:
            conformers.append((symbols, np.array(coords), energy))
    return conformers


def boltzmann_weights(energies_hartree, T=298.15):
    """Normalized Boltzmann weights from GFN2 energies (Hartree) — METADATA / QC only."""
    KCAL = 627.509; RT = 1.987e-3 * T
    e = np.asarray(energies_hartree, float) * KCAL
    mask = np.isfinite(e)
    w = np.full_like(e, np.nan, dtype=float)
    if not mask.any():
        return w
    er = e[mask] - np.nanmin(e[mask])
    raw = np.exp(-er / RT)
    w[mask] = raw / raw.sum()
    return w


def _mol_from_xyz(template, coords):
    """Embed one conformer's coords onto a bonded SMILES template (same atom order)."""
    mol = Chem.Mol(template); mol.RemoveAllConformers()
    if mol.GetNumAtoms() != len(coords):
        return None
    conf = Chem.Conformer(mol.GetNumAtoms())
    for i, (x, y, z) in enumerate(coords):
        conf.SetAtomPosition(i, (float(x), float(y), float(z)))
    mol.AddConformer(conf, assignId=True)
    return mol


def per_conf_df(run_dir, isomer, pair):
    """Per-conformer descriptor table for one isomer, over both solvents. Reads the FULL native
    ensemble (ensemble.xyz) and computes descriptors on ALL cregen conformers — no local dedup.
    `energy`/`w` are kept as metadata; `n_raw` = the full conformer count."""
    frames = []
    for solv in SOLVENTS:
        jp = run_dir / solv / "ensemble.json"
        xp = run_dir / solv / "ensemble.xyz"
        smi = json.load(open(jp)).get("smiles") if jp.exists() else None
        if not xp.exists() or smi is None:
            continue
        template = Chem.AddHs(Chem.MolFromSmiles(smi))
        confs = parse_xyz_ensemble(xp)
        n_raw = len(confs)
        w = boltzmann_weights([e for _, _, e in confs])   # metadata / QC only
        bb, rows = None, []
        for i, (syms, coords, energy) in enumerate(confs):
            mol = _mol_from_xyz(template, coords)
            if mol is None or mol.GetNumAtoms() != len(syms):
                continue
            cid = mol.GetConformer().GetId()
            if bb is None:
                bb = backbone_hbond_atoms(mol)
            try:
                sd = surface_descriptors_mol(mol, cid)
                ih = imhb_descriptors_mol(mol, cid, bb)
                rows.append(dict(
                    pair=pair, isomer=isomer, solvent=solv, w=w[i], energy=energy, n_raw=n_raw,
                    psa=sd["psa"],
                    rg=Descriptors3D.RadiusOfGyration(mol, confId=cid),
                    asph=Descriptors3D.Asphericity(mol, confId=cid),
                    sphe=Descriptors3D.SpherocityIndex(mol, confId=cid),
                    npr1=Descriptors3D.NPR1(mol, confId=cid),
                    npr2=Descriptors3D.NPR2(mol, confId=cid),
                    SA_HD=sd["hbd_sasa"], SA_HA=sd["hba_sasa"],
                    hydrophobic=sd["hydrophobic_sasa"], amphi=sd["amphi_moment"],
                    IMHB=ih["imhb"], IMHB_bb=ih["imhb_bb"], IMHB_res=ih["imhb_res"],
                    IMHBD=ih["imhbd"], IMHBA=ih["imhba"]))
            except Exception:
                pass
        frames.append(pd.DataFrame(rows))
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

print("library loaded: parse_xyz_ensemble, surface_descriptors_mol, imhb_descriptors_mol, per_conf_df")

library loaded: parse_xyz_ensemble, surface_descriptors_mol, imhb_descriptors_mol, per_conf_df


## Step 1 — point at your ensembles
Set the two run folders (each with `water/` + `mem/` holding `ensemble.sdf` + `ensemble.json`) and a label. The defaults point at an example pair in this repo; `_find` locates them whether you run from the repo root or the notebook folder. For your own data, set absolute paths.

In [2]:
from pathlib import Path

# ── edit these ─────────────────────────────────────────────────────────
PAIR       = "3-12-8-12"
RUN_DIR_R  = "results/conformers/DOPC 3-12-8-12/3-12-8-12 Xylene Linker/DOPC 3-12-8-12 R"
RUN_DIR_S  = "results/conformers/DOPC 3-12-8-12/3-12-8-12 Xylene Linker/DOPC 3-12-8-12 S"
OUT_DIR    = "results/notebook_descriptors"

def _find(path):
    """Return the path if it exists, else look for it upward from the cwd (repo-demo
    convenience). This locates example DATA only — it imports no code."""
    p = Path(path)
    if p.exists():
        return p
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / path).exists():
            return base / path
    return p

RUN_DIR_R, RUN_DIR_S = _find(RUN_DIR_R), _find(RUN_DIR_S)
OUT_DIR = Path(OUT_DIR); OUT_DIR.mkdir(parents=True, exist_ok=True)
for tag, d in [("R", RUN_DIR_R), ("S", RUN_DIR_S)]:
    ok = (d / "water" / "ensemble.json").exists() and (d / "mem" / "ensemble.json").exists()
    print(f"  {tag}: {'OK' if ok else 'MISSING water/ or mem/ ensembles'}  ({d})")

  R: OK  (C:\Users\Admin\Documents\Hu Lab\Code\Python\Chameleon_Predictor\results\conformers\DOPC 3-12-8-12\3-12-8-12 Xylene Linker\DOPC 3-12-8-12 R)
  S: OK  (C:\Users\Admin\Documents\Hu Lab\Code\Python\Chameleon_Predictor\results\conformers\DOPC 3-12-8-12\3-12-8-12 Xylene Linker\DOPC 3-12-8-12 S)


## Step 2 — per-conformer table (the file Notebook 3 reads)
For **every conformer in the full native `ensemble.xyz`** (both solvents), we rebuild the geometry on a SMILES-bonded template and compute PSA, Rg, IMHB (backbone/side-chain) and PMI shape (NPR1/NPR2). Energy and Boltzmann weight (`w`) are stored as **secondary metadata** (used only for the optional lowest-energy QC marker in Notebook 3); SMILES for Fig 1.

In [3]:
frames = []
for iso, run_dir in [("R", RUN_DIR_R), ("S", RUN_DIR_S)]:
    df = per_conf_df(run_dir, iso, PAIR)
    smi = None
    for solv in SOLVENTS:
        jp = run_dir / solv / "ensemble.json"
        if jp.exists():
            smi = json.load(open(jp)).get("smiles")
            break
    df["smiles"] = smi
    for solv in SOLVENTS:
        sub = df[df.solvent == solv]
        if len(sub):
            print(f"  {iso} {solv}: {len(sub)} conformers")
    frames.append(df)

percon = pd.concat(frames, ignore_index=True)
PERCONF_CSV = OUT_DIR / f"per_conformer_{PAIR}.csv"
percon.to_csv(PERCONF_CSV, index=False)
print(f"\nper-conformer table -> {PERCONF_CSV}   ({len(percon)} rows, {percon.shape[1]} cols)")
percon.head()

  R water: 479 conformers
  R mem: 530 conformers


  S water: 431 conformers
  S mem: 208 conformers

per-conformer table -> results\notebook_descriptors\per_conformer_3-12-8-12.csv   (1648 rows, 22 cols)


,pair,isomer,solvent,w,energy,n_raw,psa,rg,asph,sphe,...,SA_HD,SA_HA,hydrophobic,amphi,IMHB,IMHB_bb,IMHB_res,IMHBD,IMHBA,smiles
0,3-12-8-12,R,water,0.078836,-163.792601,479,224.12,4.416604,0.115452,0.639928,...,80.96,143.16,572.84,3.528,4,1,3,4,4,C#CCCC(=O)N[C@H]1CSCc2ccccc2CSC[C@@H](C(N)=O)N...
1,3-12-8-12,R,water,0.052858,-163.792224,479,229.21,4.417450,0.125474,0.615863,...,85.03,144.18,560.60,3.431,4,1,3,4,4,C#CCCC(=O)N[C@H]1CSCc2ccccc2CSC[C@@H](C(N)=O)N...
2,3-12-8-12,R,water,0.048492,-163.792142,479,205.33,4.419097,0.120075,0.636694,...,64.75,140.58,587.68,3.352,4,1,3,4,4,C#CCCC(=O)N[C@H]1CSCc2ccccc2CSC[C@@H](C(N)=O)N...
3,3-12-8-12,R,water,0.042950,-163.792028,479,226.95,4.440192,0.127346,0.586924,...,71.60,155.35,586.65,3.831,5,2,3,5,4,C#CCCC(=O)N[C@H]1CSCc2ccccc2CSC[C@@H](C(N)=O)N...
4,3-12-8-12,R,water,0.033850,-163.791803,479,222.44,4.446795,0.121096,0.590318,...,68.93,153.51,593.98,3.934,5,2,3,5,4,C#CCCC(=O)N[C@H]1CSCc2ccccc2CSC[C@@H](C(N)=O)N...


## Step 3 — geometric summary (distribution over unique conformers)
Distribution statistics per isomer / solvent / descriptor — **median, min, max, range, IQR** — the geometry-first summary that replaces Boltzmann-weighted means. `range` and `IQR` capture ensemble **flexibility**. The median water→chloroform 3D-PSA drop (**ΔPSA_median**) is the chameleonicity readout.

In [4]:
def gstats(v):
    """Geometric distribution stats over a descriptor's values (unweighted)."""
    v = np.asarray(v, float); v = v[np.isfinite(v)]
    if v.size == 0:
        return dict(n=0, median=np.nan, min=np.nan, max=np.nan, range=np.nan, iqr=np.nan)
    q1, q3 = np.percentile(v, [25, 75])
    return dict(n=int(v.size), median=round(float(np.median(v)), 2), min=round(float(v.min()), 2),
                max=round(float(v.max()), 2), range=round(float(v.max() - v.min()), 2),
                iqr=round(float(q3 - q1), 2))

rows = []
for iso in ["R", "S"]:
    for solv in ["water", "mem"]:
        for col in ["psa", "rg", "IMHB_bb"]:
            sub = percon[(percon.isomer == iso) & (percon.solvent == solv)]
            rows.append(dict(isomer=iso, solvent=solv, descriptor=col, **gstats(sub[col])))

summary = pd.DataFrame(rows)
AGG_CSV = OUT_DIR / f"descriptors_{PAIR}.csv"
summary.to_csv(AGG_CSV, index=False)
print(f"geometric summary -> {AGG_CSV}\n")

# median-based chameleonicity: water median 3D-PSA minus chloroform median 3D-PSA, per isomer
def med(iso, solv, col):
    m = summary[(summary.isomer == iso) & (summary.solvent == solv) & (summary.descriptor == col)]
    return m["median"].iloc[0] if len(m) else np.nan

for iso in ["R", "S"]:
    wm, mm = med(iso, "water", "psa"), med(iso, "mem", "psa")
    print(f"  {iso}: median 3D-PSA  water {wm} -> chloroform {mm}   (ΔPSA_median = {round(wm - mm, 2)})")

summary

geometric summary -> results\notebook_descriptors\descriptors_3-12-8-12.csv

  R: median 3D-PSA  water 238.71 -> chloroform 218.28   (ΔPSA_median = 20.43)
  S: median 3D-PSA  water 194.52 -> chloroform 173.09   (ΔPSA_median = 21.43)


,isomer,solvent,descriptor,n,median,min,max,range,iqr
0,R,water,psa,479,238.71,180.32,275.61,95.29,18.11
1,R,water,rg,479,4.46,4.28,4.71,0.43,0.09
2,R,water,IMHB_bb,479,2.00,1.00,3.00,2.00,0.00
3,R,mem,psa,530,218.28,136.00,271.13,135.13,24.60
4,R,mem,rg,530,4.72,4.44,4.96,0.52,0.11
5,R,mem,IMHB_bb,530,2.00,0.00,3.00,3.00,0.00
6,S,water,psa,431,194.52,129.43,269.29,139.86,47.50
7,S,water,rg,431,4.59,4.34,4.98,0.64,0.13
8,S,water,IMHB_bb,431,2.00,1.00,3.00,2.00,1.00
9,S,mem,psa,208,173.09,128.63,251.49,122.86,58.35


---
**Outputs (in `OUT_DIR`):**
- `per_conformer_<pair>.csv` — **Notebook 3's input** (the distribution figures build from this).
- `descriptors_<pair>.csv` — geometric summary (median / min / max / range / IQR per isomer·solvent·descriptor).

**Next:** open `03_report_figures.ipynb` and point it at the per-conformer CSV above.